# E3-J — Audit de la contribution sémantique à la préparation des données IA

**Question expérimentale :** quelles propriétés de traçabilité, de structuration et d'interopérabilité sont *effectivement démontrées* par les artefacts TalTech, indépendamment de la RMSE ?

L'audit distingue : (A) données natives et extraction RDF avec provenance ; (B) dataset supervisé et clés d'identification ; (C) preuve de correspondance IFC ; (D) relations interdispositifs. **Il ne prétend pas démontrer une amélioration de RMSE due à la sémantique**, ni mesurer un temps d'intégration évité sans protocole contrefactuel. La provenance d'un fichier source et l'alignement natif IFC sont des propriétés différentes.

Le notebook lit les CSV produits par les étapes antérieures. Il n'altère aucun résultat existant. Copier les fichiers requis dans le répertoire du projet, ou modifier `DATA_DIR` dans la première cellule.


**Version corrigée :** chemins des huit CSV définis individuellement dans `outputs/`.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, json, hashlib, platform
ROOT=Path.cwd().resolve()
# Les CSV sont répartis dans les sous-dossiers de outputs/.
# Tous les chemins sont relatifs au projet actuel, sans accès à l'ancienne copie.
OUT=ROOT/'outputs'/'E3_J_SEMANTIC_CONTRIBUTION_AUDIT'
OUT.mkdir(parents=True,exist_ok=True)
FILES={
 'traceability':Path('04_sparql/02_native_traceability_checks.csv'),
 'long':Path('04_sparql/03_ml_extraction_long_traceable.csv'),
 'supervised':Path('06_common_experiment/01_common_supervised_dataset.csv'),
 'nodes':Path('07C_graph/01_graph_node_index.csv'),
 'uuid_ifc':Path('02_reconciliation/01_uuid_ifc_direct_match.csv'),
 'ifc_evidence':Path('02_reconciliation/09_uuid_ifc_evidence.csv'),
 'native_edges':Path('07C_graph/06_native_graph_interdevice_edges.csv'),
 'quality':Path('05B_data_quality/10_data_quality_summary.csv')
}
DATA_DIR=ROOT/'outputs'
missing=[str(DATA_DIR/path) for path in FILES.values() if not (DATA_DIR/path).is_file()]
if missing:
    raise FileNotFoundError('Fichiers absents dans le projet courant :\n'+'\n'.join(missing))
print('Racine du projet :',ROOT)
for key,path in FILES.items():print(key,'->',DATA_DIR/path)


In [ ]:
trace=pd.read_csv(DATA_DIR/FILES['traceability'])
long=pd.read_csv(DATA_DIR/FILES['long'],dtype={'uuid':'string','observation':'string','point':'string'})
sup=pd.read_csv(DATA_DIR/FILES['supervised'],dtype={'uuid':'string'})
nodes=pd.read_csv(DATA_DIR/FILES['nodes'],dtype={'uuid':'string'})
uid=pd.read_csv(DATA_DIR/FILES['uuid_ifc'],dtype={'operational_uuid':'string'})
ifc=pd.read_csv(DATA_DIR/FILES['ifc_evidence'],dtype={'operational_uuid':'string'})
edges=pd.read_csv(DATA_DIR/FILES['native_edges'])
quality=pd.read_csv(DATA_DIR/FILES['quality'])
required_long={'uuid','timestamp','variable','value','unit','point','observation'}
required_sup={'uuid','timestamp','temperature','relative_humidity','co2','target_timestamp','split'}
assert required_long<=set(long.columns)
assert required_sup<=set(sup.columns)
assert {'operational_uuid','native_direct_match'}<=set(uid.columns)
assert {'node_index','uuid'}<=set(nodes.columns)
print('Long rows:',len(long),'Supervised rows:',len(sup),'Devices:',nodes.uuid.nunique())


## 1. Audit de traçabilité des observations

La présence de colonnes `point` et `observation` est vérifiée au niveau de l'extraction RDF. Les liens vers les fichiers sources (`prov:wasDerivedFrom`) sont contrôlés séparément à partir du bilan d'audit déjà produit ; ce bilan n'est pas une vérification RDF indépendante.


In [ ]:
def ratio(x,y):return float(x/y) if y else np.nan
rows=[]
def add(group,metric,value,denominator=None,interpretation=''):
    rows.append({'group':group,'metric':metric,'value':value,'denominator':denominator,
                 'ratio':ratio(value,denominator) if denominator is not None else np.nan,
                 'interpretation':interpretation})
N=len(long)
for col in ('uuid','timestamp','variable','value','unit','point','observation'):
    good=int(long[col].notna().sum())
    add('RDF extraction',f'non-null {col}',good,N)
add('RDF extraction','distinct observations',int(long.observation.nunique()),N)
add('RDF extraction','distinct semantic points',int(long.point.nunique()))
add('RDF extraction','distinct operational devices',int(long.uuid.nunique()))
add('RDF extraction','duplicate observation identifiers',int(long.observation.duplicated().sum()),N)
add('RDF extraction','duplicate uuid timestamp variable',int(long.duplicated(['uuid','timestamp','variable']).sum()),N,
    'This measures key uniqueness in extracted rows; do not assume perfect sampling.')
assert set(long.uuid.dropna().astype(str))==set(nodes.uuid.dropna().astype(str)),'Device coverage mismatch'
trace['passed_normalized']=trace.passed.astype(str).str.strip().str.lower().eq('true')
for _,r in trace.iterrows():
    add('Previously reported RDF audit',str(r['check']),str(r['actual']),None,
        'Previous audit passed='+str(bool(r['passed_normalized'])))
assert trace.passed_normalized.all(),'An earlier RDF audit reports a failure'


## 2. Dataset supervisé et correspondance d'identifiants

Le dataset supervisé n'a pas nécessairement conservé l'URI de chaque observation source. Nous vérifions donc **séparément** la présence des identifiants d'appareil et l'existence de liens observation→point dans la table extraite ; **nous ne déclarons pas une traçabilité ligne-à-ligne jusqu'au dataset IA** sans table de jointure explicite.


In [ ]:
M=len(sup)
add('Supervised dataset','rows',M)
add('Supervised dataset','distinct devices',int(sup.uuid.nunique()))
add('Supervised dataset','UUID covered by RDF extraction',int(sup.uuid.isin(long.uuid).sum()),M)
add('Supervised dataset','non-null timestamps',int(sup.timestamp.notna().sum()),M)
add('Supervised dataset','non-null target timestamps',int(sup.target_timestamp.notna().sum()),M)
add('Supervised dataset','duplicate UUID+timestamp+split',int(sup.duplicated(['uuid','timestamp','split']).sum()),M)
for v in ('temperature','relative_humidity','co2'):
    add('Supervised dataset',f'non-null {v}',int(sup[v].notna().sum()),M)
add('IFC','operational UUID with direct native IFC match',int(uid.native_direct_match.astype(str).str.lower().eq('true').sum()),len(uid),
    'Native direct match only; no inference of enriched mappings')
add('IFC','UUID with any IFC evidence row',int(ifc.ifc_global_id.notna().groupby(ifc.operational_uuid).any().sum()),len(uid))
add('Native graph','explicit interdevice edges',len(edges),None,
    'An empty file is not evidence that learned correlations or enriched edges are semantic native relations')
add('Native graph','devices',len(nodes))
report=pd.DataFrame(rows)
report.to_csv(OUT/'00_PRIMARY_J_semantic_evidence.csv',index=False)
print(report.to_string(index=False,max_rows=100))


## 3. Vérifications de cohérence et limites

Les preuves sémantiques observées doivent être distinguées des fonctionnalités **non démontrées**. En particulier : l'absence de correspondance UUID↔IFC native interdit de présenter les sept dispositifs comme directement rattachés à l'IFC ; l'absence d'arêtes natives interdit d'appeler le graphe de corrélations « graphe de relations sémantiques natives ».


In [ ]:
checks=[]
def ck(name,condition,comment):checks.append({'check':name,'passed':bool(condition),'comment':comment})
ck('7 operational devices',len(nodes)==7 and nodes.uuid.nunique()==7,'As expected in E3 experiments')
ck('RDF extracted devices match graph index',set(long.uuid.dropna().astype(str))==set(nodes.uuid.astype(str)),
   'Device-level consistency')
ck('Supervised UUID subset of RDF extracted UUID',set(sup.uuid.dropna().astype(str))<=set(long.uuid.dropna().astype(str)),
   'Does not establish row-level observation provenance')
ck('Native direct IFC matches absent',not uid.native_direct_match.astype(str).str.lower().eq('true').any(),
   'If false, inspect direct matches rather than assuming absence')
ck('No native interdevice edges',len(edges)==0,'A zero native edge count is a limitation, not a missing test')
ck('RDF audit prior checks all passed',trace.passed_normalized.all(),'Relies on earlier traceability audit')
checks=pd.DataFrame(checks)
checks.to_csv(OUT/'01_J_consistency_checks.csv',index=False)
print(checks.to_string(index=False))
if not checks.passed.all():print('ATTENTION: inspect failed checks before using the paper conclusions.')


In [ ]:
limitations=[
 ('Source provenance','Supported by previous RDF traceability audit and extraction observation URIs','Independent RDF triple verification not repeated here'),
 ('Semantic observation and point identifiers','Directly visible in RDF extraction CSV','Source observation-to-supervised-row joins not supplied'),
 ('Operational UUID↔IFC native mapping','Not supported by direct identifier audit','No native cross-system correspondence; enrichment must be labelled explicitly'),
 ('Native interdevice semantic edges','Not evidenced by native edge inventory','Learned/correlation graph is a constructed relation, not native Brick relation'),
 ('Prediction improvement from semantics','Not established by this audit','Requires a controlled counterfactual with same features/splits/model'),
 ('Time saved / interoperability gains','Not quantified','Requires measured integration effort or controlled tasks')]
pd.DataFrame(limitations,columns=['claim','evidence_status','qualification']).to_csv(
    OUT/'02_J_claim_evidence_matrix.csv',index=False)
manifest={'experiment':'E3-J semantic contribution evidence audit',
 'files':{k:str(DATA_DIR/v) for k,v in FILES.items()},
 'file_sha256':{k:hashlib.sha256((DATA_DIR/v).read_bytes()).hexdigest() for k,v in FILES.items()},
 'python':platform.python_version(),'pandas':pd.__version__,
 'design':'descriptive audit of existing semantic artefacts, not causal predictive ablation',
 'limitations':['previous RDF provenance audit not independently re-run',
 'no native IFC UUID mapping or native interdevice semantic edges',
 'no row-level RDF-to-supervised lineage established',
 'no counterfactual RMSE benefit inferred']}
(OUT/'03_J_protocol_and_hashes.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Saved audit files to:',OUT)


## Fichiers à envoyer

Compresser `outputs/E3_J_SEMANTIC_CONTRIBUTION_AUDIT/` : `00_PRIMARY_J_semantic_evidence.csv`, `01_J_consistency_checks.csv`, `02_J_claim_evidence_matrix.csv`, `03_J_protocol_and_hashes.json`.

**Conclusion attendue si les contrôles passent :** la représentation RDF soutient une extraction structurée avec identifiants de points et d'observations et une provenance rapportée par l'audit antérieur. Elle ne démontre ni la correspondance IFC native, ni des relations interdispositifs sémantiques natives, ni une réduction de RMSE. Pour prouver une amélioration de performance imputable à la sémantique, il faudrait un protocole contrefactuel distinct.
